In [0]:
CATALOG = "mvp_mmgd"
SCHEMA_BRONZE = "bronze"
VOLUME_RAW = "raw"

# Selecionar apenas a região SUL: SigUF é o nome da coluna que a ANEEL usa para o estado.
UFS_REGIAO_SUL = ["PR", "SC", "RS"]

# Caminho onde estão os dois arquivos utilizados como dataset
PATH_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/{VOLUME_RAW}"

print(PATH_VOLUME)

In [0]:
#Definindo o tipo de dados dos campos e verificando se os dados estão corretos da primeira tabela selecionada
df_empreendimento_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .option("inferSchema", "true")
    .csv(f"{PATH_VOLUME}/empreendimento-geracao-distribuida.csv")
)

df_empreendimento_raw.printSchema()

In [0]:
#Conferindo as 20 primeiras linhas da tabela
display(df_empreendimento_raw.limit(20))


In [0]:
#Definindo o tipo de dados dos campos e verificando se os dados estão corretos da segunda tabela selecionada
df_tecnica_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .option("inferSchema", "true")
    .csv(f"{PATH_VOLUME}/empreendimento-gd-informacoes-tecnicas-fotovoltaica.csv")
)

df_tecnica_raw.printSchema()
display(df_tecnica_raw.limit(20))

In [0]:
from pyspark.sql.functions import lit, current_timestamp, col

#Filtro do escopo definido nesse MVP: Selecionando apenas as regiões SUL - PR, SC e RS
df_empreendimento_sul = df_empreendimento_raw.filter(col("SigUF").isin(UFS_REGIAO_SUL))

qtd_total = df_empreendimento_raw.count()
qtd_sul = df_empreendimento_sul.count()
print(f"Quatidade Região Sul: {qtd_sul} de {qtd_total} empreendimentos ({qtd_sul/qtd_total:.1%})")

#Adicionando as colunas de controle
df_empreendimento_bronze = (
    df_empreendimento_sul
    .withColumn("_dth_ingestao", current_timestamp())
    .withColumn("_arquivo_origem", lit("empreendimento-geracao-distribuida.csv"))
    .withColumn("_camada", lit("bronze"))
)

# Salva como tabela Delta dentro do catálogo
(
    df_empreendimento_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.empreendimento_gd")
)

print("Tabela empreendimento_gd salva com sucesso")


In [0]:
#Agora é necessário selecionar apenas a região Sul da tabela de informações técnicas
#Selecionando a lista de códigos de empreendimentos que jáestão na Região Sul:

codigos_sul = df_empreendimento_sul.select("CodEmpreendimento").distinct()

#Semi Join - mantém somente as linhas da tabela de informação técnica cujo código de geração corresponde a um empreendimento do Sul. 
df_tecnica_sul = df_tecnica_raw.join(
    codigos_sul,
    df_tecnica_raw["CodGeracaoDistribuida"] == codigos_sul["CodEmpreendimento"],
    "left_semi"
)

qtd_tecnica_total = df_tecnica_raw.count()
qtd_tecnica_sul = df_tecnica_sul.count()
print(f"Quatidade Região Sul: {qtd_tecnica_sul} de {qtd_tecnica_total} empreendimentos ({qtd_tecnica_sul/qtd_tecnica_total:.1%})")

#Adicionando as colunas de controle:
df_tecnica_bronze = (
    df_tecnica_sul
    .withColumn("_dth_ingestao", current_timestamp())
    .withColumn("_arquivo_origem", lit("empreendimento-gd-informacoes-tecnicas-fotovoltaica.csv"))
    .withColumn("_camada", lit("bronze"))
)

# Salva como tabela Delta dentro do catálogo
(
    df_tecnica_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.tecnica_fotovoltaica")
)

print("Tabela tecnica_fotovoltaica salva com sucesso")





In [0]:
#Documentando as tabelas no Catalog
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{SCHEMA_BRONZE}.empreendimento_gd IS
    'Cadastro de empreendimentos de micro e minigeração distribuída. Recorte: Região Sul (PR, SC, RS). Fonte: ANEEL - Portal de Dados Abertos. Licença: ODbL. Camada: Bronze.'
""")

spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{SCHEMA_BRONZE}.tecnica_fotovoltaica IS
    'Informações técnicas dos empreendimentos fotovoltaicos (módulos, inversores). Recorte: Região Sul (PR, SC, RS). Fonte: ANEEL - Portal de Dados Abertos. Licença: ODbL. Camada: Bronze.'
""")

print("Descrições registradas no Unity Catalog.")

In [0]:
for tabela in ["empreendimento_gd", "tecnica_fotovoltaica"]:
    nome_completo = f"{CATALOG}.{SCHEMA_BRONZE}.{tabela}"
    print(f"\n=== {nome_completo} ===")
    spark.table(nome_completo).printSchema()

display(spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.empreendimento_gd").limit(5))

In [0]:
from datetime import datetime

log_ingestao = [
    {
        "tabela": "empreendimento_gd",
        "url_origem": "https://dadosabertos.aneel.gov.br/dataset/5e0fafd2-21b9-4d5b-b622-40438d40aba2/resource/b1bd71e7-d0ad-4214-9053-cbd58e9564a7/download/empreendimento-geracao-distribuida.csv",
        "metodo_ingestao": "upload manual via interface do Databricks",
        "qtd_linhas": qtd_sul,
        "dth_ingestao": datetime.now(),
    },
    {
        "tabela": "tecnica_fotovoltaica",
        "url_origem": "https://dadosabertos.aneel.gov.br/dataset/5e0fafd2-21b9-4d5b-b622-40438d40aba2/resource/49fa9ca0-f609-4ae3-a6f7-b97bd0945a3a/download/empreendimento-gd-informacoes-tecnicas-fotovoltaica.csv",
        "metodo_ingestao": "upload manual via interface do Databricks",
        "qtd_linhas": qtd_tecnica_sul,
        "dth_ingestao": datetime.now(),
    },
]

df_log = spark.createDataFrame(log_ingestao)

(
    df_log.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}._log_ingestao")
)

display(df_log)